In [1]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain.tools import tool

# Load environment variables from .env file
load_dotenv()

# Configure OpenRouter model
# IMPORTANT: API key is loaded from .env file (see .env.example)
model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="gpt-oss-120b:free",  # Different free model
    temperature=0.1,
    max_tokens=1000,
)

In [2]:
@tool
def convert_temperature(value: float, from_unit: str, to_unit: str) -> float:
    """
    Convert temperature between Celsius and Fahrenheit.
    
    Args:
        value: The temperature value to convert
        from_unit: Source unit ("celsius" or "fahrenheit")
        to_unit: Target unit ("celsius" or "fahrenheit")
    
    Returns:
        Converted temperature value
    """
    if from_unit.lower() == "celsius" and to_unit.lower() == "fahrenheit":
        return (value * 9/5) + 32
    elif from_unit.lower() == "fahrenheit" and to_unit.lower() == "celsius":
        return (value - 32) * 5/9
    elif from_unit.lower() == to_unit.lower():
        return value
    else:
        raise ValueError(f"Unsupported conversion from {from_unit} to {to_unit}")

In [3]:
agent = create_agent(
    model=model,
    system_prompt="You are a helpful assistant that can convert temperatures between Celsius and Fahrenheit. " \
    "Always use the convert_temperature tool when users ask for temperature conversions.",
    tools=[convert_temperature]
)

In [4]:
response = agent.invoke({"messages": [HumanMessage("Convert 30 degrees Celsius to Fahrenheit.")]})
print(response["messages"][-1].content)

30 °C is equal to **86 °F**.
